# Tutorial 5: Fitting Individual Stars with BruteForce

This tutorial demonstrates how to fit individual stars using brutus's BruteForce class, which performs fast Bayesian inference over pre-computed model grids.

## Topics Covered

1. **Data preparation** (flux units, parallaxes, coordinates)
2. **BruteForce class** setup and configuration
3. **Running fits** with various options
4. **Result visualization** and interpretation
5. **Model comparison** (MIST vs Bayestar)

## Prerequisites

This tutorial requires the following brutus data files:
- `grid_mist_v9.h5` - MIST model grid
- `offsets_mist_v8.txt` - Photometric offsets
- `Orion_l204.7_b-19.2.h5` - Example Orion field data
- `bayestar2019_v1.h5` (optional) - 3D dust map
- `grid_bayestar_v5.h5` (optional) - Bayestar empirical grid

If you don't have these files, run the optional download cell below.

In [ ]:
# Optional: Download required data files (only run if needed)
# Uncomment the lines below to download

# from brutus.data import fetch_grids, fetch_dustmaps
# fetch_grids(target_dir='../data/DATAFILES/')  # Downloads model grids
# fetch_dustmaps(target_dir='../data/DATAFILES/')  # Downloads Bayestar dust maps

# Note: The Orion field data needs to be downloaded separately from the brutus repository

In [ ]:
# Imports and setup
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

from tutorial_utils import (
    setup_tutorial,
    find_brutus_data_file,
    save_figure as _save_fig,
    print_section,
    load_orion_data,
)

info = setup_tutorial(5, title="Tutorial 05: Fitting Individual Stars with BruteForce")
plots_dir = info['plot_dir']


def save_figure(fig, name):
    """Save figure to this tutorial's plot directory."""
    _save_fig(fig, 5, name)

## Section 1: Data Preparation for BruteForce

BruteForce requires specific data formats:
- **Photometry in flux units** (maggies = 10^(-0.4 * magnitude))
- **Parallaxes in mas** (milliarcseconds) from Gaia
- **Galactic coordinates** (l, b) for applying priors

Let's load and examine the Orion field data.

In [ ]:
from brutus.utils import inv_magnitude, magnitude

# Load example Orion field data
print("Loading Orion field data...")
data = load_orion_data()

print(f"\nLoaded {len(data['phot'])} sources")
print(f"  Photometry shape: {data['phot'].shape}")
print(f"  Filters: Pan-STARRS (grizy) + 2MASS (JHKs) = 8 bands")
print(f"  Valid parallaxes: {np.sum(np.isfinite(data['parallax']))}")
print(f"  Coordinate range: l=[{data['coords'][:, 0].min():.1f}, {data['coords'][:, 0].max():.1f}], "
      f"b=[{data['coords'][:, 1].min():.1f}, {data['coords'][:, 1].max():.1f}] deg")

# Show example conversion from magnitude to flux
example_mags = np.array([15.0, 18.0, 20.0, 22.0])
example_flux = 10**(-0.4 * example_mags)

print("\nMagnitude to Flux Conversion Examples:")
for mag, flux in zip(example_mags, example_flux):
    print(f"  mag = {mag:.1f} -> flux = {flux:.2e} maggies")

In [ ]:
# Data format demonstration
fig, axes = plt.subplots(2, 3, figsize=(15, 10))

# Panel 1: Magnitude vs flux conversion
ax = axes[0, 0]
mags = np.linspace(10, 25, 100)
flux = 10**(-0.4 * mags)  # Maggies

ax.semilogy(mags, flux, 'b-', lw=2)
ax.set_xlabel('Magnitude')
ax.set_ylabel('Flux (maggies)')
ax.set_title('Magnitude to Flux Conversion')
ax.grid(True, alpha=0.3)

# Add reference points
for mag, color in [(15, 'green'), (20, 'orange'), (25, 'red')]:
    ax.scatter(mag, 10**(-0.4 * mag), s=100, color=color, zorder=5, edgecolor='black', linewidth=1)
    ax.annotate(f'mag={mag}\n{10**(-0.4 * mag):.2e}', 
                (mag, 10**(-0.4 * mag)),
                xytext=(2, 0), textcoords='offset points', fontsize=8)

# Panel 2: Parallax distribution
ax = axes[0, 1]
valid_plx = np.isfinite(data['parallax']) & (data['parallax'] > 0)
ax.hist(data['parallax'][valid_plx], bins=50, alpha=0.7, color='green', edgecolor='darkgreen')
ax.axvline(1.0, color='red', ls='--', label='1 mas = 1 kpc', lw=2)
ax.set_xlabel('Parallax (mas)')
ax.set_ylabel('Number of Sources')
ax.set_title('Parallax Distribution')
ax.legend()
ax.grid(True, alpha=0.3, axis='y')

# Panel 3: Sky distribution
ax = axes[0, 2]
scatter = ax.scatter(data['coords'][:, 0], data['coords'][:, 1],
                     s=1, alpha=0.5, c=np.sum(data['mask'], axis=1),
                     cmap='viridis')
ax.set_xlabel('Galactic l (deg)')
ax.set_ylabel('Galactic b (deg)')
ax.set_title('Sky Distribution')
plt.colorbar(scatter, ax=ax, label='Bands detected')
ax.grid(True, alpha=0.3)

# Panel 4: Band coverage
ax = axes[1, 0]
band_coverage = np.sum(data['mask'], axis=0) / len(data['mask']) * 100

band_names = ['g', 'r', 'i', 'z', 'y', 'J', 'H', 'Ks']
colors_band = ['blue', 'green', 'orange', 'red', 'darkred', 'purple', 'brown', 'black']

bars = ax.bar(range(len(band_names)), band_coverage, color=colors_band, alpha=0.7)
ax.set_xticks(range(len(band_names)))
ax.set_xticklabels(band_names)
ax.set_ylabel('Coverage (%)')
ax.set_title('Band Coverage')
ax.grid(True, alpha=0.3, axis='y')

# Add values on bars
for bar, val in zip(bars, band_coverage):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 1,
            f'{val:.0f}%', ha='center', fontsize=9)

# Panel 5: Error distribution (SNR)
ax = axes[1, 1]

# Calculate SNR for each measurement
snr = data['phot'] / data['err']
snr_flat = snr[data['mask']].flatten()
snr_flat = snr_flat[np.isfinite(snr_flat) & (snr_flat > 0)]

ax.hist(snr_flat, bins=np.logspace(0, 3, 50), alpha=0.7, color='orange', edgecolor='darkorange')
ax.axvline(5, color='red', ls='--', label='SNR=5 threshold', lw=2)
ax.set_xscale('log')
ax.set_xlabel('Signal-to-Noise Ratio')
ax.set_ylabel('Number of Measurements')
ax.set_title('Photometric SNR Distribution')
ax.legend()
ax.grid(True, alpha=0.3)

# Panel 6: Data requirements summary
ax = axes[1, 2]
ax.axis('off')

summary_text = """
BruteForce Data Requirements:

Required arrays (N_sources x N_bands):
  - phot: flux in maggies
  - err: flux errors in maggies
  - mask: boolean (True if observed)

Optional arrays (N_sources):
  - parallax: in mas
  - parallax_err: in mas
  - coords: [l, b] in degrees

Quality recommendations:
  - SNR > 5 for key bands
  - Parallax SNR > 5 if available
  - >= 3 bands per source
"""

ax.text(0.05, 0.95, summary_text, transform=ax.transAxes,
       fontsize=10, va='top', family='monospace')

plt.suptitle('Data Preparation for BruteForce', fontsize=16, fontweight='bold')
save_figure(fig, 'data_preparation')
plt.show()

print("\nData preparation analysis complete")
print("  Key points:")
print("  - Convert magnitudes to maggies (flux units)")
print("  - Check parallax quality (SNR > 5)")
print("  - Ensure proper coordinate system (Galactic l, b)")

## Section 2: BruteForce Class Setup

BruteForce performs fast stellar parameter inference by:
1. **Evaluating likelihood** over a pre-computed grid of stellar models
2. **Marginalizing analytically** over nuisance parameters (distance, extinction)
3. **Incorporating priors** (Galactic structure, 3D dust, parallax)

Let's load the MIST model grid and initialize BruteForce.

In [ ]:
from brutus.analysis import BruteForce
from brutus.core import StarGrid
from brutus.data import load_models, filters

# Load MIST model grid
print("Loading MIST model grid...")
grid_file = find_brutus_data_file('grid_mist_v9.h5')

# Define filters (Pan-STARRS + 2MASS)
filt = filters.ps[:-2] + filters.tmass  # Skip PS w and open bands
print(f"\nUsing filters: {filt}")

# Load models
models, labels, label_mask = load_models(grid_file, filters=filt)

print(f"\nLoaded {len(models):,} models")
print(f"  Model shape: {models.shape}")
print(f"  Parameter fields: {list(labels.dtype.names)}")
print(f"  Key parameters:")
print(f"    Mass range: [{labels['mini'].min():.2f}, {labels['mini'].max():.1f}] M_sun")
print(f"    [Fe/H] range: [{labels['feh'].min():.2f}, {labels['feh'].max():.2f}]")
print(f"    log(age) range: [{labels['loga'].min():.2f}, {labels['loga'].max():.2f}]")

# Create StarGrid and initialize BruteForce
grid = StarGrid(models, labels)
bf = BruteForce(grid)

print(f"\nBruteForce initialized with {len(filt)} bands")

In [ ]:
# Visualize the fitting framework
fig, axes = plt.subplots(2, 3, figsize=(15, 10))

# Panel 1: Fitting workflow
ax = axes[0, 0]
ax.axis('off')

workflow = """
BruteForce Workflow:

1. For each model in grid:
   -> Fit (distance, A_V, R_V)
   -> Calculate chi-squared

2. Apply priors:
   -> Galactic structure
   -> 3D dust maps
   -> Parallax constraint

3. Marginalize:
   -> Sample from posterior
   -> Weight by probability

4. Output:
   -> Parameter samples
   -> Best-fit values
   -> Uncertainties
"""

ax.text(0.05, 0.95, workflow, transform=ax.transAxes,
       fontsize=11, va='top', family='monospace')

# Panel 2: Grid coverage in CMD
ax = axes[0, 1]

# Sample subset for plotting
n_plot = min(10000, len(models))
idx_plot = np.random.choice(len(models), n_plot, replace=False)

# Calculate colors and magnitudes at 1 kpc
g_r = models[idx_plot, 0, 0] - models[idx_plot, 1, 0]
g_mag = models[idx_plot, 0, 0]

scatter = ax.scatter(g_r, g_mag, s=0.5, alpha=0.3,
                    c=labels[idx_plot]['feh'], cmap='RdYlBu_r',
                    vmin=-2, vmax=0.5)
ax.set_xlabel('g - r (mag)')
ax.set_ylabel('g (mag at 1 kpc)')
ax.set_title('MIST Grid Coverage in CMD')
ax.invert_yaxis()
ax.set_xlim(-0.5, 3)
ax.set_ylim(25, -5)
plt.colorbar(scatter, ax=ax, label='[Fe/H]')
ax.grid(True, alpha=0.3)

# Panel 3: Parameter distributions
ax = axes[0, 2]
ax.hist(labels['mini'], bins=np.logspace(-1, 2, 50),
       alpha=0.7, color='blue', edgecolor='darkblue')
ax.set_xlabel('Initial Mass (M_sun)')
ax.set_ylabel('Number of Models')
ax.set_title('Mass Distribution in Grid')
ax.set_xscale('log')
ax.set_yscale('log')
ax.grid(True, alpha=0.3)

# Panel 4: Analytical marginalization concept
ax = axes[1, 0]

# Create fake 2D likelihood surface
test_dist = np.linspace(0.1, 10, 100)
test_av = np.linspace(0, 3, 100)
D, A = np.meshgrid(test_dist, test_av)
likelihood = np.exp(-0.5 * ((D - 2)**2 / 0.5**2 + (A - 0.5)**2 / 0.2**2))

im = ax.imshow(likelihood, origin='lower', aspect='auto',
              extent=[test_dist.min(), test_dist.max(),
                     test_av.min(), test_av.max()],
              cmap='YlOrRd')
ax.set_xlabel('Distance (kpc)')
ax.set_ylabel('A(V) (mag)')
ax.set_title('Analytical Marginalization')
plt.colorbar(im, ax=ax, label='Likelihood')

# Add contours
ax.contour(D, A, likelihood, levels=5, colors='white', alpha=0.3, linewidths=0.5)

# Panel 5: Fitting options
ax = axes[1, 1]
ax.axis('off')

options_text = """
Key Fitting Options:

Priors:
  - phot_offsets: band zeropoints
  - data_coords: (l,b) for Galactic prior
  - dustfile: 3D dust map

Sampling:
  - Ndraws: posterior samples
  - Nmc_prior: MC prior samples
  - weights: source weights

Performance:
  - running_io: real-time output
  - pool: multiprocessing
  - batch_size: memory control

Likelihood:
  - logl_dim_prior: chi2 vs Gaussian
  - error_floor: systematic floor
"""

ax.text(0.05, 0.95, options_text, transform=ax.transAxes,
       fontsize=10, va='top', family='monospace')

# Panel 6: Output structure
ax = axes[1, 2]
ax.axis('off')

output_text = """
BruteForce Output (HDF5):

Per-source arrays:
  - model_idx: grid indices
  - obj_chi2min: best chi2
  - obj_Nbands: bands used

Posterior samples:
  - samps_dist: distance
  - samps_red: A(V)
  - samps_dred: R(V)
  - samps_logp: log posterior

Maximum likelihood:
  - ml_dist, ml_av, ml_rv
  - ml_cov_sar: covariance

Metadata:
  - filters: band names
  - runtime: timing info
"""

ax.text(0.05, 0.95, output_text, transform=ax.transAxes,
       fontsize=10, va='top', family='monospace')

plt.suptitle('BruteForce Fitting Framework', fontsize=16, fontweight='bold')
save_figure(fig, 'bruteforce_setup')
plt.show()

print("\nBruteForce setup complete")

## Section 3: Running BruteForce Fits

Now let's fit a subset of sources from the Orion field, demonstrating various fitting options.

Key considerations:
- **Photometric offsets** calibrate models to match observed photometry
- **3D dust maps** provide extinction priors
- **Parallax constraints** from Gaia improve distance estimates
- **Galactic priors** based on position help constrain parameters

In [ ]:
# Load photometric offsets (calibration corrections)
from brutus.data import load_offsets

print("Loading photometric offsets...")
try:
    offsets_file = find_brutus_data_file('offsets_mist_v8.txt')
    offsets_mist = load_offsets(offsets_file, filters=filt)
    print("\nPhotometric offsets (multiplicative flux):")
    for band, offset in zip(filt, offsets_mist):
        print(f"  {band}: {offset:.4f}")
except FileNotFoundError:
    offsets_mist = np.ones(len(filt))
    print("  Offsets file not found, using unity offsets")

# Select subset for fitting (sources with good data)
good_sources = np.sum(data['mask'], axis=1) >= 5  # At least 5 bands
has_parallax = np.isfinite(data['parallax']) & (data['parallax'] > 0)
idx_fit = np.where(good_sources & has_parallax)[0][:50]  # First 50 good sources

print(f"\nSelected {len(idx_fit)} sources for fitting")
print(f"  Criteria: >=5 bands detected AND valid parallax")
print(f"  Average bands per source: {np.mean(np.sum(data['mask'][idx_fit], axis=1)):.1f}")

# Try to load 3D dust map
try:
    dustfile = find_brutus_data_file('bayestar2019_v1.h5')
    print("\n3D dust map (Bayestar) available")
except FileNotFoundError:
    dustfile = None
    print("\n3D dust map not available (will use flat prior)")

In [ ]:
# Run BruteForce fits
output_file = plots_dir / 'orion_fits_mist.h5'

print(f"Running BruteForce fits on {len(idx_fit)} sources...")

bf.fit(
    data=data['phot'][idx_fit],
    data_err=data['err'][idx_fit],
    data_mask=data['mask'][idx_fit],
    data_labels=idx_fit,
    save_file=str(output_file),
    data_coords=data['coords'][idx_fit],
    parallax=data['parallax'][idx_fit],
    parallax_err=data['parallax_err'][idx_fit],
    phot_offsets=offsets_mist,
    dustfile=dustfile,
    Ndraws=100,
    Nmc_prior=20,
    logl_dim_prior=True,
    save_dar_draws=True,
    verbose=True
)

print(f"\nFitting complete! Results saved to {output_file}")

In [ ]:
# Load and analyze results
import h5py

with h5py.File(output_file, 'r') as f:
    chi2 = f['obj_chi2min'][:]
    nbands = f['obj_Nbands'][:]
    model_idx = f['model_idx'][:]
    ml_scale = f['ml_scale'][:]
    ml_av = f['ml_av'][:]
    ml_rv = f['ml_rv'][:]
    log_post = f['obj_log_post'][:]
    dists = f['samps_dist'][:]
    reds = f['samps_red'][:]
    dreds = f['samps_dred'][:]

print("Loaded fitting results:")
print(f"  chi2 values: {chi2.shape}")
print(f"  Model indices: {model_idx.shape}")
print(f"  ML scale/AV/RV: {ml_scale.shape}")
print(f"  Distance samples: {dists.shape}")

# Create fit quality visualization
fig, axes = plt.subplots(2, 3, figsize=(15, 10))

# Panel 1: Chi2 distribution
ax = axes[0, 0]
chi2_per_band = chi2 / nbands
ax.hist(chi2_per_band, bins=30, alpha=0.7, color='blue', edgecolor='darkblue')
ax.axvline(1.0, color='red', ls='--', lw=2, label='Expected (chi2/n = 1)')
ax.axvline(np.median(chi2_per_band), color='green', ls='--', lw=2, label=f'Median = {np.median(chi2_per_band):.2f}')
ax.set_xlabel('chi2/band')
ax.set_ylabel('Number of Sources')
ax.set_title('Goodness of Fit')
ax.legend()
ax.grid(True, alpha=0.3)

# Panel 2: Distance distribution
ax = axes[0, 1]
mean_dists = np.mean(dists, axis=1)
ax.hist(mean_dists, bins=30, alpha=0.7, color='green', edgecolor='darkgreen')
ax.axvline(np.median(mean_dists), color='red', ls='--', lw=2, label=f'Median = {np.median(mean_dists):.2f} kpc')
ax.set_xlabel('Distance (kpc)')
ax.set_ylabel('Number of Sources')
ax.set_title('Distance Distribution')
ax.legend()
ax.grid(True, alpha=0.3)

# Panel 3: Extinction distribution
ax = axes[0, 2]
mean_av = np.mean(reds, axis=1)
ax.hist(mean_av, bins=30, alpha=0.7, color='brown', edgecolor='darkred')
ax.axvline(np.median(mean_av), color='blue', ls='--', lw=2, label=f'Median = {np.median(mean_av):.2f} mag')
ax.set_xlabel('A(V) (mag)')
ax.set_ylabel('Number of Sources')
ax.set_title('Extinction Distribution')
ax.legend()
ax.grid(True, alpha=0.3)

# Panel 4: Distance vs extinction
ax = axes[1, 0]
scatter = ax.scatter(mean_dists, mean_av, alpha=0.5, s=20,
                    c=chi2_per_band, cmap='viridis', vmin=0, vmax=3)
ax.set_xlabel('Distance (kpc)')
ax.set_ylabel('A(V) (mag)')
ax.set_title('Distance-Extinction Relation')
plt.colorbar(scatter, ax=ax, label='chi2/band')
ax.grid(True, alpha=0.3)

# Panel 5: Stellar parameters (using best-fit model per source)
ax = axes[1, 1]

best_draw = np.argmax(log_post, axis=1)
best_model_per_source = model_idx[np.arange(len(chi2)), best_draw]
unique_masses = []
for idx in best_model_per_source:
    if idx < len(labels):
        unique_masses.append(labels[idx]['mini'])

if unique_masses:
    ax.hist(unique_masses, bins=np.logspace(-0.5, 1, 30),
           alpha=0.7, color='purple', edgecolor='darkviolet')
    ax.set_xlabel('Initial Mass (M_sun)')
    ax.set_ylabel('Number of Sources')
    ax.set_title('Mass Distribution')
    ax.set_xscale('log')
    ax.grid(True, alpha=0.3)

# Panel 6: Fit statistics
ax = axes[1, 2]
ax.axis('off')

# Calculate statistics
good_fits = chi2_per_band < 3
stats_text = f"""
Fit Statistics:

Total sources: {len(chi2)}
Good fits (chi2/n < 3): {good_fits.sum()} ({100*good_fits.sum()/len(chi2):.1f}%)

Distance:
  Mean: {np.mean(mean_dists):.2f} kpc
  Median: {np.median(mean_dists):.2f} kpc
  Std: {np.std(mean_dists):.2f} kpc

Extinction:
  Mean: {np.mean(mean_av):.2f} mag
  Median: {np.median(mean_av):.2f} mag
  Std: {np.std(mean_av):.2f} mag

chi2/band:
  Mean: {np.mean(chi2_per_band):.2f}
  Median: {np.median(chi2_per_band):.2f}

R(V):
  Mean: {np.mean(dreds):.2f}
  Std: {np.std(dreds):.2f}
"""

ax.text(0.05, 0.95, stats_text, transform=ax.transAxes,
       fontsize=11, va='top', family='monospace')

plt.suptitle('BruteForce Fitting Results', fontsize=16, fontweight='bold')
save_figure(fig, 'fitting_results')
plt.show()

print("\nResult analysis complete")

## Section 4: Visualizing Individual Results

Let's examine individual star fits in detail to understand the posterior distributions and parameter correlations.

In [ ]:
# Select example stars with different properties
examples = [
    np.argmin(chi2/nbands),  # Best fit
    np.argmax(mean_av),  # High extinction  
    np.argmin(mean_dists),  # Nearby
    len(chi2)//2  # Typical
]

example_names = ['Best Fit', 'High Extinction', 'Nearby', 'Typical']

print("Selected example stars:")
for idx, name in zip(examples, example_names):
    print(f"  {name}: Star {idx}")
    print(f"    χ²/n = {chi2[idx]/nbands[idx]:.2f}")
    print(f"    Distance = {mean_dists[idx]:.2f} kpc")
    print(f"    A(V) = {mean_av[idx]:.2f} mag")

In [ ]:
# Create detailed visualization for example stars
fig = plt.figure(figsize=(20, 16))

for panel_idx, (star_idx, star_name) in enumerate(zip(examples, example_names)):
    # Get data for this star
    data_idx = idx_fit[star_idx]
    
    # Find the best-fit draw (highest log-posterior)
    ibest = np.argmax(log_post[star_idx])
    best_midx = model_idx[star_idx, ibest]
    best_scale = ml_scale[star_idx, ibest]
    best_av = ml_av[star_idx, ibest]
    best_rv = ml_rv[star_idx, ibest]
    best_dist = 1.0 / np.sqrt(np.abs(best_scale)) if best_scale > 0 else np.nan
    
    # SED plot
    ax1 = plt.subplot(4, 4, panel_idx*4 + 1)
    
    if best_midx < len(models):
        # Compute reddened model magnitudes using grid coefficients
        mag0 = models[best_midx, :, 0]   # unreddened mag at 1 kpc
        R0   = models[best_midx, :, 1]   # reference reddening coefficient
        dR   = models[best_midx, :, 2]   # differential reddening coefficient
        rvec = R0 + best_rv * dR          # total reddening vector
        model_mags = mag0 + best_av * rvec  # apply extinction
        
        # Distance modulus: models stored at 1 kpc (scale = 1/d_kpc^2)
        if np.isfinite(best_dist) and best_dist > 0:
            model_mags = model_mags + 5 * np.log10(best_dist)
        
        # Convert data to magnitudes
        from brutus.utils import magnitude
        data_mags, data_err = magnitude(
            data['phot'][data_idx],
            data['err'][data_idx]
        )
        
        # Approximate effective wavelengths (um) for PS1 grizy + 2MASS JHKs
        wavelengths = np.array([0.48, 0.62, 0.75, 0.87, 0.96, 1.25, 1.65, 2.17])
        
        # Plot data and model
        mask_star = data['mask'][data_idx]
        ax1.errorbar(wavelengths[mask_star], data_mags[mask_star],
                    yerr=data_err[mask_star],
                    fmt='ko', label='Data', capsize=3, markersize=6)
        ax1.plot(wavelengths, model_mags, 'r-', label='Model', lw=2, alpha=0.7)
        ax1.plot(wavelengths, model_mags, 'ro', markersize=4, alpha=0.5)
        
        ax1.set_xlabel('Wavelength (um)')
        ax1.set_ylabel('Magnitude')
        ax1.invert_yaxis()
        ax1.legend(fontsize=8, loc='upper right')
        ax1.set_title(f'{star_name}: chi2/n = {chi2[star_idx]/nbands[star_idx]:.2f}',
                     fontsize=10)
        ax1.grid(True, alpha=0.3)
    
    # Distance-extinction posterior
    ax2 = plt.subplot(4, 4, panel_idx*4 + 2)
    
    H, xedges, yedges = np.histogram2d(
        dists[star_idx], reds[star_idx],
        bins=25, density=True
    )
    extent = [xedges[0], xedges[-1], yedges[0], yedges[-1]]
    
    im = ax2.imshow(H.T, origin='lower', extent=extent,
              aspect='auto', cmap='YlOrRd', interpolation='gaussian')
    ax2.contour(H.T, extent=extent, levels=5, colors='white', alpha=0.3, linewidths=0.5)
    
    ax2.set_xlabel('Distance (kpc)')
    ax2.set_ylabel('A(V) (mag)')
    ax2.set_title('Distance-Extinction Posterior', fontsize=10)
    ax2.grid(True, alpha=0.3)
    
    # Show parallax constraint if available
    if np.isfinite(data['parallax'][data_idx]) and data['parallax'][data_idx] > 0:
        plx_dist = 1.0 / data['parallax'][data_idx]  # kpc (parallax in mas)
        plx_err = plx_dist**2 * data['parallax_err'][data_idx]
        ax2.axvline(plx_dist, color='blue', ls='--', alpha=0.7, lw=2, label='Parallax')
        ax2.axvspan(plx_dist - plx_err, plx_dist + plx_err, alpha=0.2, color='blue')
        ax2.legend(fontsize=8)
    
    # Mark best-fit point on distance-extinction plane
    if np.isfinite(best_dist):
        ax2.plot(best_dist, best_av, 'w*', markersize=12, markeredgecolor='k', zorder=10)
    
    # Parameter distributions
    ax3 = plt.subplot(4, 4, panel_idx*4 + 3)
    
    params_sample = []
    for idx in model_idx[star_idx][:50]:
        if idx < len(labels):
            params_sample.append([
                labels[idx]['mini'],
                labels[idx]['feh'],
                10**labels[idx]['loga'] / 1e9
            ])
    
    if params_sample:
        params_sample = np.array(params_sample)
        
        ax3.hist(params_sample[:, 0], bins=20, alpha=0.7,
                color='blue', edgecolor='darkblue')
        ax3.axvline(np.median(params_sample[:, 0]), color='red', ls='--', lw=2)
        ax3.set_xlabel('Initial Mass (M_sun)')
        ax3.set_ylabel('Samples')
        ax3.set_title('Mass Distribution', fontsize=10)
        ax3.grid(True, alpha=0.3)
    
    # Summary statistics
    ax4 = plt.subplot(4, 4, panel_idx*4 + 4)
    ax4.axis('off')
    
    if params_sample is not None and len(params_sample) > 0:
        mass_str = f"{np.median(params_sample[:, 0]):.2f} +/- {np.std(params_sample[:, 0]):.2f}"
        feh_str = f"{np.median(params_sample[:, 1]):.2f} +/- {np.std(params_sample[:, 1]):.2f}"
        age_str = f"{np.median(params_sample[:, 2]):.1f} +/- {np.std(params_sample[:, 2]):.1f}"
    else:
        mass_str = feh_str = age_str = "N/A"
    
    # Best-fit model parameters
    if best_midx < len(labels):
        bf_mass = labels[best_midx]['mini']
        bf_feh = labels[best_midx]['feh']
        bf_loga = labels[best_midx]['loga']
    else:
        bf_mass = bf_feh = bf_loga = np.nan
    
    summary = f"""
    {star_name} Summary:
    
    Best-fit model:
    Distance: {best_dist:.2f} kpc
    A(V): {best_av:.2f} mag
    R(V): {best_rv:.2f}
    Mass: {bf_mass:.2f} M_sun
    [Fe/H]: {bf_feh:.2f}
    log(age): {bf_loga:.2f}

    Posterior (median +/- std):
    Mass: {mass_str} M_sun
    [Fe/H]: {feh_str}
    Age: {age_str} Gyr
    
    chi2: {chi2[star_idx]:.1f}  ({nbands[star_idx]} bands)
    chi2/n: {chi2[star_idx]/nbands[star_idx]:.2f}
    """
    
    ax4.text(0.05, 0.95, summary, transform=ax4.transAxes,
            fontsize=9, va='top', family='monospace')

plt.suptitle('Individual Star Fitting Results', fontsize=16, fontweight='bold')
plt.tight_layout(rect=[0, 0, 1, 0.96])
save_figure(fig, 'individual_results')
plt.show()

print("\nIndividual result visualization complete")

## Section: Photometric Likelihood Functions

Before running full BruteForce fits, it can be useful to understand the underlying
likelihood functions that drive the inference. brutus provides three key functions
for evaluating how well model SEDs match observed photometry:

- **`phot_loglike`**: The standard photometric log-likelihood. Given observed fluxes,
  errors, and a set of model fluxes, it computes the Gaussian log-likelihood
  (optionally with a chi-square dimensionality prior that penalizes models based
  on goodness-of-fit relative to the number of observed bands).

- **`chisquare_outlier_loglike`**: An outlier model based on the chi-square
  distribution. It computes the log-likelihood at a p-value threshold, representing
  the probability that an object is an outlier (i.e., not well described by any
  model in the grid). This is the default outlier model when `logl_dim_prior=True`.

- **`uniform_outlier_loglike`**: An outlier model assuming a quasi-uniform
  distribution within +/- sigma_clip standard deviations of the data. This is the
  default outlier model when `logl_dim_prior=False`.

During BruteForce fitting, the best-fit model likelihood is compared against the
outlier likelihood to flag problematic sources. Below we demonstrate these functions
on simple synthetic data.

In [ ]:
# Photometric likelihood functions: standalone demonstration
# These functions work on synthetic data -- no external files needed.

from brutus.utils import phot_loglike
from brutus.utils.photometry import chisquare_outlier_loglike, uniform_outlier_loglike

np.random.seed(42)

# --- Create synthetic data ---
# A "true" SED in 5 filters (flux in maggies, roughly mag 15-18 range)
true_sed = np.array([1.0e-6, 8.0e-7, 6.0e-7, 4.0e-7, 2.5e-7])  # 5 filters
Nfilt = len(true_sed)

# Observed data: true SED + Gaussian noise (SNR ~ 20 per band)
flux_err = true_sed / 20.0  # 5% errors
obs_flux = true_sed + np.random.randn(Nfilt) * flux_err

# Shape arrays for phot_loglike: (Nobj, Nfilt)
data_flux = obs_flux[np.newaxis, :]       # (1, 5)
data_err = flux_err[np.newaxis, :]        # (1, 5)
data_mask = np.ones_like(data_flux)       # (1, 5) -- all bands valid

# --- Build a grid of offset models ---
# Vary an overall scale factor around the true SED to simulate distance offsets
Nmodels = 200
scale_factors = np.linspace(0.5, 1.5, Nmodels)
model_grid = true_sed[np.newaxis, :] * scale_factors[:, np.newaxis]  # (Nmodels, 5)
model_grid_3d = model_grid[np.newaxis, :, :]  # (1, Nmodels, 5) for phot_loglike

# --- Evaluate standard log-likelihood (with and without dim_prior) ---
lnl_standard = phot_loglike(data_flux, data_err, model_grid_3d,
                            mask=data_mask, dim_prior=False)  # (1, Nmodels)
lnl_dprior = phot_loglike(data_flux, data_err, model_grid_3d,
                          mask=data_mask, dim_prior=True)     # (1, Nmodels)

# --- Evaluate outlier likelihoods ---
lnl_chisq_outlier = chisquare_outlier_loglike(data_flux, data_err)    # (1,)
lnl_unif_outlier = uniform_outlier_loglike(data_flux, data_err)       # (1,)

print("Photometric Likelihood Comparison")
print("=" * 55)
print(f"  Number of filters:          {Nfilt}")
print(f"  Number of model grid points: {Nmodels}")
print(f"  Best standard lnL:          {lnl_standard[0].max():.2f}")
print(f"  Best dim-prior lnL:         {lnl_dprior[0].max():.2f}")
print(f"  Chi-square outlier lnL:     {lnl_chisq_outlier[0]:.2f}")
print(f"  Uniform outlier lnL:        {lnl_unif_outlier[0]:.2f}")
print()
print("Interpretation:")
print("  If best-model lnL >> outlier lnL  --> source is well-fit")
print("  If best-model lnL ~  outlier lnL  --> source may be an outlier")

# --- Plot ---
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

# Panel 1: Standard vs dim-prior log-likelihood curves
ax = axes[0]
ax.plot(scale_factors, lnl_standard[0], 'b-', lw=2, label='Standard')
ax.plot(scale_factors, lnl_dprior[0], 'r-', lw=2, label='With dim_prior')
ax.axvline(1.0, color='grey', ls=':', lw=1, label='True scale')
ax.set_xlabel('Scale Factor (model / true)')
ax.set_ylabel('Log-Likelihood')
ax.set_title('Log-Likelihood vs Model Scale')
ax.legend()
ax.grid(True, alpha=0.3)

# Panel 2: Likelihood curves with outlier thresholds
ax = axes[1]
ax.plot(scale_factors, lnl_dprior[0], 'r-', lw=2, label='dim_prior lnL')
ax.axhline(lnl_chisq_outlier[0], color='darkorange', ls='--', lw=2,
           label=f'Chi-sq outlier ({lnl_chisq_outlier[0]:.1f})')
ax.axhline(lnl_unif_outlier[0], color='green', ls='--', lw=2,
           label=f'Uniform outlier ({lnl_unif_outlier[0]:.1f})')
ax.set_xlabel('Scale Factor (model / true)')
ax.set_ylabel('Log-Likelihood')
ax.set_title('Model vs Outlier Likelihoods')
ax.legend(fontsize=9)
ax.grid(True, alpha=0.3)

# Panel 3: Synthetic SED with best-fit model
ax = axes[2]
wave = np.arange(1, Nfilt + 1)  # filter index as proxy for wavelength
best_idx = np.argmax(lnl_standard[0])
ax.errorbar(wave, obs_flux * 1e6, yerr=flux_err * 1e6,
            fmt='ko', capsize=4, markersize=8, label='Observed', zorder=5)
ax.plot(wave, true_sed * 1e6, 's--', color='grey', ms=7,
        label='True SED', alpha=0.6)
ax.plot(wave, model_grid[best_idx] * 1e6, 'r^-', ms=7,
        label=f'Best-fit (scale={scale_factors[best_idx]:.2f})')
ax.set_xlabel('Filter Index')
ax.set_ylabel('Flux (micro-maggies)')
ax.set_title('Synthetic SED Comparison')
ax.legend(fontsize=9)
ax.grid(True, alpha=0.3)

plt.suptitle('Photometric Likelihood Functions', fontsize=14, fontweight='bold')
plt.tight_layout(rect=[0, 0, 1, 0.95])
save_figure(fig, 'likelihood_comparison')
plt.show()

print("\nPlot saved.")

## Section: Advanced BruteForce Options

`BruteForce.fit()` accepts many parameters beyond the basics shown earlier.
Understanding these options lets you tune performance, control output size,
and adjust the statistical model to match your science case.

The parameters fall into several categories:

- **Likelihood control**: `logl_dim_prior` switches between a standard Gaussian
  log-likelihood and a chi-square dimensionality-corrected version that penalizes
  over-fitting (or under-fitting) relative to the number of observed bands.

- **Prior adjustments**: `apply_agewt` adds an age-based weight from the MIST
  evolutionary tracks (accounting for time spent in each evolutionary phase).
  `apply_grad` applies a gradient-based correction for non-uniform grid spacing.
  `av_gauss` lets you impose a Gaussian prior on A(V) instead of using the
  distance-reddening relation from a 3D dust map.

- **Output thresholds**: `wt_thresh` and `cdf_thresh` control which models are
  kept in the posterior sampling. Raising these thresholds reduces output file
  size at the cost of potentially trimming the tails of the posterior.

The cell below prints a reference table of these advanced parameters. It does
**not** run a fit (which would be too slow for an interactive tutorial).

In [ ]:
# Advanced BruteForce.fit() parameter reference
# This cell documents parameters -- it does NOT run a fit.

advanced_params = [
    # (parameter, default, description)
    ("logl_dim_prior", "True",
     "Use chi-square dimensionality prior on the likelihood.\n"
     "True  -> penalizes models based on chi2 relative to DOF (recommended).\n"
     "False -> standard Gaussian log-likelihood (no DOF correction)."),

    ("apply_agewt", "True",
     "Apply age-based weighting from MIST evolutionary tracks.\n"
     "Accounts for the different amounts of time a star spends\n"
     "in each evolutionary phase (e.g., main sequence vs. giant branch)."),

    ("apply_grad", "True",
     "Apply gradient-based correction for non-uniform grid spacing.\n"
     "Corrects for the fact that model grid points are not evenly\n"
     "distributed in parameter space (mass, age, metallicity)."),

    ("av_gauss", "None",
     "Gaussian prior on A(V) as a (mean, std) tuple.\n"
     "Example: av_gauss=(0.5, 0.2) centers extinction at A(V)=0.5 mag.\n"
     "If None, uses the distance-reddening relation from the dust map."),

    ("wt_thresh", "1e-3",
     "Weight threshold for model selection: keeps models with\n"
     "weight > wt_thresh * max(weight). Lower values keep more\n"
     "models (broader posterior tails) but increase output size."),

    ("cdf_thresh", "2e-3",
     "CDF threshold for model selection. Models contributing less\n"
     "than this fraction of cumulative probability are discarded.\n"
     "Used as fallback if wt_thresh is None."),

    ("Ndraws", "250",
     "Number of posterior samples saved per object.\n"
     "More draws give smoother posteriors but larger output files.\n"
     "Typical range: 100 (quick) to 500 (publication quality)."),

    ("Nmc_prior", "50",
     "Number of Monte Carlo samples for prior integration.\n"
     "Higher values improve accuracy of the distance/extinction\n"
     "marginalization at the cost of computation time."),

    ("rv_gauss", "(3.32, 0.18)",
     "Gaussian prior on R(V) as a (mean, std) tuple.\n"
     "Default based on Schlafly et al. (2016).\n"
     "Set to None to use a flat prior within rvlim bounds."),

    ("running_io", "True",
     "If True, writes each object to disk immediately after fitting.\n"
     "Safer for long runs (recoverable on crash) but slower on\n"
     "network filesystems. Set False to batch-write at the end."),
]

# Print formatted table
print("Advanced BruteForce.fit() Parameters")
print("=" * 72)

for param, default, desc in advanced_params:
    print(f"\n  {param}  (default: {default})")
    print("  " + "-" * 68)
    for line in desc.split("\n"):
        print(f"    {line}")

print("\n" + "=" * 72)
print("\nExample usage with advanced options:")
print("""
    bf.fit(
        data=flux, data_err=flux_err, data_mask=mask,
        data_labels=obj_ids, save_file='results.h5',
        # --- advanced options ---
        logl_dim_prior=True,      # recommended for robust fitting
        apply_agewt=True,         # weight by evolutionary timescales
        apply_grad=True,          # correct for grid spacing
        av_gauss=(0.3, 0.1),     # informative A(V) prior
        rv_gauss=(3.32, 0.18),   # Schlafly+2016 R(V) prior
        wt_thresh=1e-3,           # keep top 0.1% of models
        cdf_thresh=2e-3,          # CDF cutoff
        Ndraws=250,               # posterior samples per source
        Nmc_prior=50,             # MC prior samples
        running_io=True,          # write results incrementally
    )
""")

## Summary and Key Takeaways

This tutorial has demonstrated how to fit individual stars using BruteForce:

### Key Steps

1. **Data Preparation**
   - Convert magnitudes to flux (maggies)
   - Ensure proper error arrays and masks
   - Include parallax and coordinates when available

2. **Model Setup**
   - Load pre-computed model grids (MIST or Bayestar)
   - Apply photometric offsets for calibration
   - Initialize BruteForce with models

3. **Running Fits**
   - Configure priors (Galactic, dust, parallax)
   - Set sampling parameters (Ndraws, Nmc_prior)
   - Save results to HDF5 for analysis

4. **Result Analysis**
   - Examine χ² distributions for fit quality
   - Analyze parameter distributions and correlations
   - Visualize individual SEDs and posteriors

### Best Practices

- **Data Quality**: Require SNR > 5 and ≥3 bands per source
- **Photometric Offsets**: Always apply appropriate calibrations
- **Priors**: Use 3D dust maps and parallax when available
- **Validation**: Check χ²/n values (should be ~1 for good fits)
- **Uncertainties**: Use posterior samples for proper error estimation

### Model Selection

- **MIST**: Best for evolved stars, wide parameter space, stellar parameters
- **Bayestar**: Best for main sequence, fast surveys, distance/reddening focus

### Next Steps

- **Tutorial 6**: Cluster Analysis and Population Fitting
- **Tutorial 7**: 3D Dust Mapping
- **Tutorial 8**: Photometric Calibration

In [ ]:
print("Tutorial 5 Complete!")
print("="*60)
print("\nGenerated plots:")
for plot_file in sorted(plots_dir.glob('*.png')):
    print(f"  - {plot_file.name}")
    
print("\nOutput files:")
for h5_file in sorted(plots_dir.glob('*.h5')):
    print(f"  - {h5_file.name}")